# Prepare the official geometry split



## Configuration

The  inputs are eight geometry variables plus angle of attack. The approved targets are `cl`, `cd`, and `cm`. Learned preprocessing remains inside each model Pipeline.

In [1]:
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd

project_dir = Path.cwd()
if not (project_dir / "data" / "hiliftaeroml").is_dir():
    project_dir = project_dir.parent
assert (project_dir / "data" / "hiliftaeroml").is_dir(), "Project root not found"

data_dir = project_dir / "data" / "hiliftaeroml"
geometry_path = data_dir / "geo_values_all.csv"
forces_path = data_dir / "force_mom_all.csv"
manifest_path = data_dir / "splits" / "manifest.json"
output_dir = project_dir / "data" / "prepared"

geometry_features = [
    "IB_Flap_Deflection",
    "OB_Flap_Deflection",
    "IB_Flap_Gap_Multiplier",
    "OB_Flap_Gap_Multiplier",
    "IB_Slat_Deflection",
    "OB_Slat_Deflection",
    "IB_Slat_Gap_Multiplier",
    "OB_Slat_Gap_Multiplier",
]
feature_columns = geometry_features + ["AoA"]
target_columns = ["cl", "cd", "cm"]
expected_aoas = [4, 6, 8, 10, 12, 14, 16, 18, 20, 22]
expected_geometry_counts = {"train": 126, "validation": 18, "test": 36}
expected_row_counts = {name: count * len(expected_aoas) for name, count in expected_geometry_counts.items()}

print(f"Python {platform.python_version()} | pandas {pd.__version__} | NumPy {np.__version__}")
print("Features:", feature_columns)
print("Targets:", target_columns)

Python 3.10.20 | pandas 2.3.3 | NumPy 1.26.4
Features: ['IB_Flap_Deflection', 'OB_Flap_Deflection', 'IB_Flap_Gap_Multiplier', 'OB_Flap_Gap_Multiplier', 'IB_Slat_Deflection', 'OB_Slat_Deflection', 'IB_Slat_Gap_Multiplier', 'OB_Slat_Gap_Multiplier', 'AoA']
Targets: ['cl', 'cd', 'cm']


## Load the official split and raw tables

Only the three `geometry` split lists are used. Manifest case IDs are parsed into `GeoID` and `AoA`; no split is regenerated or randomized.

In [2]:
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_keys = {
    "geometry_train": "train",
    "geometry_val": "validation",
    "geometry_test": "test",
}

assignment_parts = []
case_pattern = r"^geo_(?P<GeoID>LHC[0-9]{3})_AoA_(?P<AoA>[0-9]{1,2})$"
for manifest_key, split_name in manifest_keys.items():
    assert manifest_key in manifest, f"Missing manifest key: {manifest_key}"
    case_ids = pd.Series(manifest[manifest_key], name="case_id", dtype="string")
    parsed = case_ids.str.extract(case_pattern)
    assert parsed.notna().all().all(), f"Malformed case ID in {manifest_key}"
    parsed["AoA"] = parsed["AoA"].astype("int64")
    part = pd.concat([case_ids, parsed], axis=1)
    part["split"] = split_name
    rebuilt = "geo_" + part["GeoID"] + "_AoA_" + part["AoA"].astype(str)
    assert rebuilt.equals(part["case_id"]), f"Non-canonical case ID in {manifest_key}"
    assignment_parts.append(part)

assignments = pd.concat(assignment_parts, ignore_index=True)
geometry = pd.read_csv(geometry_path, encoding="utf-8-sig", dtype={"GeoID": "string"})

force_columns = pd.read_csv(forces_path, nrows=0).columns.tolist()
required_force_columns = ["GeoID", "AoA", *target_columns]
assert set(required_force_columns).issubset(force_columns), "Required aerodynamic columns are missing"

# Read the required raw fields as strings. Build and validate identifiers for
# all cases before selecting which rows may materialize numeric targets.
raw_forces = pd.read_csv(
    forces_path,
    usecols=required_force_columns,
    dtype={column: "string" for column in required_force_columns},
)
raw_forces["AoA"] = pd.to_numeric(raw_forces["AoA"], errors="raise").astype("int64")
raw_forces.insert(
    0,
    "case_id",
    "geo_" + raw_forces["GeoID"] + "_AoA_" + raw_forces["AoA"].astype(str),
)

print(
    f"Loaded {len(geometry)} geometries, {len(raw_forces)} aerodynamic case identifiers, "
    f"and {len(assignments)} manifest assignments."
)


Loaded 180 geometries, 1800 aerodynamic case identifiers, and 1800 manifest assignments.


In [3]:
assert list(geometry.columns) == ["GeoID", *geometry_features]
assert geometry["GeoID"].str.fullmatch(r"LHC[0-9]{3}").all()
assert raw_forces["GeoID"].str.fullmatch(r"LHC[0-9]{3}").all()
assert all(pd.api.types.is_numeric_dtype(geometry[column]) for column in geometry_features)
assert geometry.notna().all().all() and raw_forces.notna().all().all() and assignments.notna().all().all()
assert geometry["GeoID"].is_unique
assert raw_forces["case_id"].is_unique
assert assignments["case_id"].is_unique
assert not raw_forces.duplicated(["GeoID", "AoA"]).any()
assert set(assignments["case_id"]) == set(raw_forces["case_id"])
assert set(assignments["GeoID"]) == set(geometry["GeoID"]) == set(raw_forces["GeoID"])

observed_rows = assignments["split"].value_counts().to_dict()
observed_geometries = assignments.groupby("split")["GeoID"].nunique().to_dict()
assert observed_rows == expected_row_counts
assert observed_geometries == expected_geometry_counts
assert assignments.groupby("GeoID")["split"].nunique().eq(1).all()
assert assignments.groupby("GeoID")["AoA"].apply(lambda values: sorted(values.tolist()) == expected_aoas).all()
assert raw_forces.groupby("GeoID")["AoA"].apply(lambda values: sorted(values.tolist()) == expected_aoas).all()

geometry_sets = {
    split_name: set(assignments.loc[assignments["split"].eq(split_name), "GeoID"])
    for split_name in expected_geometry_counts
}
assert geometry_sets["train"].isdisjoint(geometry_sets["validation"])
assert geometry_sets["train"].isdisjoint(geometry_sets["test"])
assert geometry_sets["validation"].isdisjoint(geometry_sets["test"])

base_inputs = assignments.merge(
    geometry, on="GeoID", how="left", validate="many_to_one", indicator=True
)
assert base_inputs["_merge"].eq("both").all()
base_inputs = base_inputs.drop(columns="_merge")

development_ids = assignments.loc[
    assignments["split"].isin(["train", "validation"]),
    ["case_id", "GeoID", "AoA"],
]
development_targets = development_ids.merge(
    raw_forces,
    on=["case_id", "GeoID", "AoA"],
    how="left",
    validate="one_to_one",
    indicator=True,
)
assert development_targets["_merge"].eq("both").all()
development_targets = development_targets.drop(columns="_merge")
for target in target_columns:
    development_targets[target] = pd.to_numeric(
        development_targets[target], errors="raise"
    )
assert np.isfinite(development_targets[target_columns].astype("float64").to_numpy()).all()

prepared_development = base_inputs.loc[
    base_inputs["split"].isin(["train", "validation"])
].merge(
    development_targets,
    on=["case_id", "GeoID", "AoA"],
    how="left",
    validate="one_to_one",
    indicator=True,
)
assert prepared_development["_merge"].eq("both").all()
prepared_development = prepared_development.drop(columns="_merge")

print("Identifier, trajectory, geometry-separation, and development-target checks passed.")


Identifier, trajectory, geometry-separation, and development-target checks passed.


## Write prepared datasets

Training and validation retain the three targets. Test output contains identifiers and inputs only.

In [4]:
identifier_columns = ["case_id", "GeoID", "AoA", "split"]
input_columns = [*identifier_columns, *geometry_features]
modeling_columns = [*input_columns, *target_columns]

train = prepared_development.loc[
    prepared_development["split"].eq("train"), modeling_columns
].sort_values(["GeoID", "AoA"]).reset_index(drop=True)
validation = prepared_development.loc[
    prepared_development["split"].eq("validation"), modeling_columns
].sort_values(["GeoID", "AoA"]).reset_index(drop=True)
test_inputs = base_inputs.loc[
    base_inputs["split"].eq("test"), input_columns
].sort_values(["GeoID", "AoA"]).reset_index(drop=True)

assert len(train) == expected_row_counts["train"]
assert len(validation) == expected_row_counts["validation"]
assert len(test_inputs) == expected_row_counts["test"]
assert set(target_columns).isdisjoint(test_inputs.columns)
assert train["split"].eq("train").all()
assert validation["split"].eq("validation").all()
assert test_inputs["split"].eq("test").all()

output_dir.mkdir(parents=True, exist_ok=True)
train.to_csv(output_dir / "train.csv", index=False, lineterminator="\n")
validation.to_csv(output_dir / "validation.csv", index=False, lineterminator="\n")
test_inputs.to_csv(output_dir / "test_inputs.csv", index=False, lineterminator="\n")

print("Wrote:")
for path in [output_dir / "train.csv", output_dir / "validation.csv", output_dir / "test_inputs.csv"]:
    print(" -", path.relative_to(project_dir).as_posix())


Wrote:
 - data/prepared/train.csv
 - data/prepared/validation.csv
 - data/prepared/test_inputs.csv


## Verify saved contract

In [5]:
saved = {
    "train": pd.read_csv(output_dir / "train.csv", dtype={"GeoID": "string"}),
    "validation": pd.read_csv(output_dir / "validation.csv", dtype={"GeoID": "string"}),
    "test": pd.read_csv(output_dir / "test_inputs.csv", dtype={"GeoID": "string"}),
}
assert list(saved["train"].columns) == modeling_columns
assert list(saved["validation"].columns) == modeling_columns
assert list(saved["test"].columns) == input_columns
assert set(target_columns).isdisjoint(saved["test"].columns)

summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": len(frame),
            "geometries": frame["GeoID"].nunique(),
            "columns": len(frame.columns),
            "missing_values": int(frame.isna().sum().sum()),
            "unique_case_ids": frame["case_id"].is_unique,
        }
        for split_name, frame in saved.items()
    ]
)
assert summary["missing_values"].eq(0).all()
assert summary["unique_case_ids"].all()
summary

,split,rows,geometries,columns,missing_values,unique_case_ids
0,train,1260,126,15,0,True
1,validation,180,18,15,0,True
2,test,360,36,12,0,True
